# Project 03 — IFRS 9 Stress Testing & Expected Credit Loss (ECL)

**CreditRiskCourse / 01_microcredit_portfolio**

## Objective
Apply a forward-looking macroeconomic overlay (2025 Recession) to the PD model and calculate the IFRS 9 Expected Credit Loss (ECL) for the portfolio. 

## IFRS 9 Staging Rules
- **Stage 1:** 12-month ECL (Performing)
- **Stage 2:** Lifetime ECL (Significant Increase in Credit Risk - SICR)
- **Stage 3:** Lifetime ECL (Credit Impaired / Defaulted)

## The Recession Scenario (2025 H1)
- Unemployment spikes by +4.5%
- Repo rate spikes by +2.5%
- GDP drops by -2.0%


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE = Path(r"C:\CreditRiskCourse\01_microcredit_portfolio")
PQ   = BASE / "data/raw/parquet"
FIG  = BASE / "outputs/figures"
SUM  = BASE / "outputs/summaries"
FIG.mkdir(parents=True, exist_ok=True)
SUM.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("Environment ready.")

## 1. Load Data & Define the Stress Scenario

We pull the latest snapshot of the portfolio (Month 36 — just before the 2025 recession hits) and calculate our Base vs. Stressed PDs.


In [ ]:
# Load the monthly performance and macro data
perf = pd.read_parquet(PQ / "loan_monthly_performance.parquet")
macro = pd.read_parquet(PQ / "macro_scenarios.parquet")

# We take a snapshot of the portfolio at Month 36 (Dec 2024, just before the recession)
snapshot_month = 36
df = perf[perf["month_id"] == snapshot_month].copy()

print(f"Snapshot Month: {snapshot_month} (Total Loans: {len(df):,})")
print(f"Total Outstanding Balance: ${df['outstanding_balance'].sum():,.2f}")

# Show the macro shock we are applying
shock = macro[macro["month_id"].between(37, 43)]
print("\n2025 Recession Macro Averages:")
print(f"Unemployment: {shock['unemployment'].mean():.2f}% (Base was ~32.5%)")
print(f"Repo Rate:    {shock['repo_rate'].mean():.2f}% (Base was ~8.25%)")
print(f"GDP Growth:   {shock['gdp_growth_yoy'].mean():.2f}%")

## 2. Apply Forward-Looking PD Overlay

To calculate IFRS 9 ECL, we must apply a **Point-in-Time (PIT)** adjustment to our base PDs. 
We derive a Macro Multiplier based on the recession severity:
\[
\text{Macro Multiplier} = \exp(0.08 \times \Delta\text{Unemployment}) \times \exp(-0.15 \times \Delta\text{GDP})
\]
With Unemployment +4.5% and GDP -2.0%, this creates a significant multiplier.


In [ ]:
# Calculate the macro multiplier
delta_unemployment = 4.5
delta_gdp = -2.0

macro_multiplier = np.exp(0.08 * delta_unemployment) * np.exp(-0.15 * delta_gdp)
print(f"Calculated Macro Multiplier: {macro_multiplier:.4f}x")
print(f"This means all PDs will increase by {(macro_multiplier - 1)*100:.1f}% during the recession.")

# For simplicity, we assume the model's true_annual_pd is our Base PD. 
# In a real bank, we would use the model's predicted PD. 
# We apply the multiplier and cap it at 1.0 (100%).
df["base_pd"] = df["true_annual_pd"].clip(0, 1)
df["stressed_pd"] = (df["base_pd"] * macro_multiplier).clip(0, 1)

print(f"\nAverage Base PD:     {df['base_pd'].mean():.2%}")
print(f"Average Stressed PD: {df['stressed_pd'].mean():.2%}")

## 3. IFRS 9 Staging & EAD

- **Stage 1:** Current (DPD = 0), 12-month ECL
- **Stage 2:** Watchlist (DPD 1-89 or Restructured), Lifetime ECL (we use 36 months)
- **Stage 3:** Credit Impaired (Defaulted), Lifetime ECL (we use 12 months LGD immediately)

We will assume a conservative **LGD of 45%** for the portfolio (standard for unsecured SME lending, though our DGP had ~70%, banks often hold collateral).


In [ ]:
# Define LGD and EAD
LGD = 0.45
df["ead"] = df["outstanding_balance"]

# Map States to IFRS 9 Stages
def assign_stage(row):
    if row["dpd_bucket"] == 4: return 3
    elif row["dpd_bucket"] >= 1 or row["restructure_flag"] == 1: return 2
    else: return 1

df["stage"] = df.apply(assign_stage, axis=1)

print("Stage Distribution (Month 36):")
print(df["stage"].value_counts().sort_index().to_string())

# Calculate ECL based on Stage
def calc_ecl(row, pd_col):
    if row["stage"] == 1:
        return row[pd_col] * LGD * row["ead"] * (12/12) # 12-month
    elif row["stage"] == 2:
        return row[pd_col] * LGD * row["ead"] * (36/12) # 36-month lifetime approximation
    else: # Stage 3
        return 1.0 * LGD * row["ead"] # Defaulted

df["ecl_base"] = df.apply(lambda r: calc_ecl(r, "base_pd"), axis=1)
df["ecl_stressed"] = df.apply(lambda r: calc_ecl(r, "stressed_pd"), axis=1)

print(f"\nTotal EAD (Exposure): ${df['ead'].sum():,.2f}")

## 4. ECL Results (Base vs. Stressed)

Here is the financial impact of the recession on the loan book.


In [ ]:
summary = df.groupby("stage").agg(
    n_loans=("loan_id", "count"),
    total_ead=("ead", "sum"),
    ecl_base=("ecl_base", "sum"),
    ecl_stressed=("ecl_stressed", "sum")
).reset_index()

summary["ecl_delta"] = summary["ecl_stressed"] - summary["ecl_base"]
summary["coverage_base_%"] = (summary["ecl_base"] / summary["total_ead"]) * 100
summary["coverage_stressed_%"] = (summary["ecl_stressed"] / summary["total_ead"]) * 100

print(summary.to_string())
summary.to_csv(SUM / "03_ECL_summary.csv", index=False)

total_base = summary["ecl_base"].sum()
total_stressed = summary["ecl_stressed"].sum()
print(f"\n--- Total Portfolio ECL ---")
print(f"Base Scenario:     ${total_base:,.2f}")
print(f"Stressed Scenario: ${total_stressed:,.2f}")
print(f"Increase:          ${total_stressed - total_base:,.2f} ({(total_stressed/total_base - 1)*100:.1f}%)")

## 5. Visualizing the Capital Impact

Banks must hold capital against this ECL. Here we plot the shift in provisions by stage.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
summary.set_index("stage")[["ecl_base", "ecl_stressed"]].plot(
    kind="bar", ax=ax, color=["#4C72B0", "#C44E52"]
)
ax.set_title("IFRS 9 Expected Credit Loss (ECL) by Stage: Base vs. Recession")
ax.set_ylabel("ECL ($)")
ax.set_xlabel("IFRS 9 Stage")
ax.set_xticklabels(["Stage 1", "Stage 2", "Stage 3"], rotation=0)
plt.tight_layout()
plt.savefig(FIG / "03_ecl_by_stage.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Significant Increase in Credit Risk (SICR) Migration

Under IFRS 9, a loan moves from Stage 1 to Stage 2 if its PD has increased significantly (SICR), even if it hasn't missed a payment yet.
We assume the bank's SICR threshold is: **PD has doubled**, OR it is 30+ DPD.

Let's look at how many Stage 1 loans migrate to Stage 2 *purely because of the macro recession*.


In [ ]:
# SICR rule for macro overlay: Stressed PD > 2 * Base PD
sicr_mask = (df["stage"] == 1) & (df["stressed_pd"] > (df["base_pd"] * 2.0))
migrated_count = sicr_mask.sum()

print(f"Stage 1 Loans migrating to Stage 2 due to SICR: {migrated_count:,} ({(migrated_count/len(df))*100:.1f}% of portfolio)")
print(f"EAD associated with SICR migration: ${df.loc[sicr_mask, 'ead'].sum():,.2f}")

# Recalculate ECL assuming these migrate to Stage 2 (Lifetime ECL)
df["ecl_post_sicr"] = df["ecl_stressed"] # Copy
# Apply lifetime (36m) formula to the migrated loans
df.loc[sicr_mask, "ecl_post_sicr"] = df.loc[sicr_mask, "stressed_pd"] * LGD * df.loc[sicr_mask, "ead"] * 3.0

total_post_sicr = df["ecl_post_sicr"].sum()
print(f"\nRevised Total ECL after SICR Migration: ${total_post_sicr:,.2f}")
print(f"Additional ECL purely from SICR staging: ${total_post_sicr - total_stressed:,.2f}")

## 7. Conclusion for Model Validation & Governance

This notebook demonstrates the full IFRS 9 Forward-Looking ECL workflow:
1. **Macro Overlay:** Applied a systemic shock (GDP drop, Unemployment spike).
2. **PD Adjustment:** Multiplied base PDs by a macro factor.
3. **Staging:** Assigned 12-month vs Lifetime ECL based on DPD and SICR.
4. **Capital Impact:** Quantified the exact Dollar increase in provisions.

**Interview Talking Point:** 
When asked about IFRS 9, you can explain that ECL is not just a historical calculation. You simulated a recession, applied it to your PD model, and measured the capital required to survive it. This is exactly the kind of advisory work Deloitte RFR does for its banking clients.
